## Example 1: Call the Web API (against a local stand-in for slack.com)

## Example 2: Block Kit layouts as Python objects

In [ ]:
from slack_sdk.models.blocks import ActionsBlock, ButtonElement, DividerBlock, MarkdownTextObject, PlainTextObject, SectionBlock

# Example 2: blocks validate their fields and render to the JSON Slack expects
blocks = [
    SectionBlock(text=MarkdownTextObject(text="*Build finished* in 3m"), block_id="status"),
    DividerBlock(),
    ActionsBlock(elements=[ButtonElement(text=PlainTextObject(text="Rerun"), action_id="rerun", value="42", style="primary")]),
]
payload = [b.to_dict() for b in blocks]
assert payload[0] == {"type": "section", "block_id": "status", "text": {"type": "mrkdwn", "text": "*Build finished* in 3m"}}, payload[0]
assert payload[2]["elements"][0]["action_id"] == "rerun" and payload[2]["elements"][0]["style"] == "primary"
print("Example 2 OK:", [p["type"] for p in payload])

## Example 3: Verify a request signed by Slack

## Example 4: API errors raise SlackApiError

In [ ]:
import http.server
import json
import threading

from slack_sdk import WebClient
from slack_sdk.errors import SlackApiError


class Api(http.server.BaseHTTPRequestHandler):
    def do_POST(self):
        self.rfile.read(int(self.headers["Content-Length"] or 0)); self.send_response(200); self.end_headers()
        self.wfile.write(json.dumps({"ok": False, "error": "channel_not_found"}).encode())
    def log_message(self, *args):
        pass


# Example 4: an {"ok": false} reply raises, carrying the error code
server = http.server.HTTPServer(("127.0.0.1", 0), Api); threading.Thread(target=server.serve_forever, daemon=True).start()
try:
    WebClient(token="xoxb-test", base_url=f"http://127.0.0.1:{server.server_port}/api/").conversations_info(channel="C404")
    raise AssertionError("no error raised")
except SlackApiError as error:
    assert error.response["error"] == "channel_not_found", error.response
    print("Example 4 OK:", error.response["error"])
server.shutdown()

## Example 5: The asyncio client (aiohttp)

In [ ]:
from aiohttp import web
from slack_sdk.web.async_client import AsyncWebClient

# Example 5: AsyncWebClient sends the same calls with aiohttp; here to a local server
async def auth_test(request):
    return web.json_response({"ok": True, "user": "bot", "team": "T1", "auth": request.headers["Authorization"]})
app = web.Application(); app.router.add_post("/api/auth.test", auth_test)
runner = web.AppRunner(app); await runner.setup(); site = web.TCPSite(runner, "127.0.0.1", 0); await site.start()
port = site._server.sockets[0].getsockname()[1]
response = await AsyncWebClient(token="xoxb-async", base_url=f"http://127.0.0.1:{port}/api/").auth_test()
await runner.cleanup()
assert response["ok"] and response["user"] == "bot" and response["auth"] == "Bearer xoxb-async", response.data
print("Example 5 OK:", response["team"])